# EPOCH 5th Datathon — 팀 오감자 · 최종 제출(B31) 코드

- 최종 제출 파일: `submit_b31_v5fix_S16k_r06_x_b1.csv` (본선 리더보드 PR-AUC 0.7466)
- 이 노트북은 원자료에서 최종 제출 파일까지 한 번에 다시 만듭니다. 시드는 모두 고정입니다.

## 구조

```
E (LightGBM 분류) ─┐
8/25 공식 라벨 ────┼─→ 방법 ① B1 ───────────────────────┐
8/25~31 영상 성장 ─┘                                    ├─→ 순위 평균(0.5 : 0.5) → B31
급등 크기 추정 ─→ 방법 ② 구조 시뮬레이터(공식 정의 view_5d) ─┘
```

| 단계 | 내용 | 절 |
|---|---|---|
| E | 과거 8개 기준일로 학습한 LightGBM (입력 28개, 기준일 안 순위 정규화) | 2 |
| B1 | `rank(E) + 0.5×(8/25 라벨) − 0.25×rank(8/25~31 영상의 성장)` | 3 |
| view_5d 재계산 | 코드북 정의(업로드 후 108~132시간 구간 스냅샷 최댓값)로 다시 계산 | 4 |
| 급등 사전분포 | 채널 특성 + 기존 영상 배율 → 채널별 급등 크기 예상값 | 5 |
| 시뮬레이터 | 9/1~9/26 영상을 16,000×4회 생성, 8/25 라벨과 맞는 세계만 남겨 P(9/1 Rising) 집계 | 6 |
| 결합 | 두 방법의 순위를 반반 평균 | 7 |

## 실행 방법

작업 폴더에 아래가 있어야 합니다(노트북을 그 폴더에서 실행).

```
epoch_data/epoch_data/*.csv      # 배포 데이터
video_snapshots.parquet          # 배포 video_snapshots (csv.gz 를 parquet 으로 바꾼 것)
```

- 필요한 패키지: pandas, numpy, lightgbm, scikit-learn, pyarrow
- 1절의 `%%writefile` 셀은 모듈 4개(`epoch_pipeline_local.py`, `structsim.py`, `simp.py`, `make_b26_funcs.py`)를 작업 폴더에 씁니다. 팀 폴더의 파일과 같은 내용입니다.
- 결과는 `submissions/nb_*.csv` 로 저장합니다(기존 제출 파일을 덮어쓰지 않음).
- 전체 실행 시간은 약 2~5분입니다.

## 데이터 사용에 대해 밝혀 둘 점

- **9/1 이후에 관측된 값:** 8/27~8/31 업로드 영상의 `view_5d`(배포 표에 있는 값, 최대 9/7 관측)를 시뮬레이터·급등 크기 추정·B1 보정항에 씁니다. 운영진 답변(10/2): 과거 구간의 값이며 사용 금지 언급 없음.
- **8/25 공식 라벨**을 직접 씁니다. 운영진 답변: 각 팀 판단, 실격 사유 아님.
- **스냅샷은 8/31 23:59까지만** 씁니다(코드에 assert).
- 외부 데이터는 쓰지 않습니다.

In [1]:
import os, time, warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
T0 = time.time(); WORK = os.getcwd()
for p in ['epoch_data/epoch_data/video_5d_views.csv', 'epoch_data/epoch_data/videos.csv', 'epoch_data/epoch_data/train_labels.csv',
          'epoch_data/epoch_data/sample_submission.csv', 'video_snapshots.parquet']:
    assert os.path.exists(os.path.join(WORK, p)), f'없음: {p}'
os.makedirs('submissions', exist_ok=True)
print('작업 폴더 확인 완료')

작업 폴더 확인 완료


## 1. 공용 모듈

팀 폴더의 모듈을 그대로 싣습니다. 뒤의 실행 셀이 import 해서 씁니다.

### 1-1. `epoch_pipeline_local.py` — 기본 피처와 E(LightGBM) 파이프라인

- `label`: 자체 라벨(공식 8/11 라벨과 98.7% 일치), `feats`·`feats2`: 기준일 D 이전 정보로 만든 입력 28개
- `build`: 기준일별 라벨·피처, `predict_all`: 기준일 안 순위 정규화 + 최근 기준일 가중(×0.7) LightGBM

In [2]:
%%writefile epoch_pipeline_local.py
# ==========================================================================
# EPOCH 5th Datathon — 팀 오감자 · 로컬 실행판 (원본: epoch_pipeline.py, Colab용)
# 원본 대비 바뀐 점
#   1) 경로: Colab glob → 이 파일이 있는 폴더(팀 폴더) 기준
#   2) 미래정보 assert: 모든 피처 입력이 기준일 D 이전인지 검사
#   3) 실험용 설정(CFG): 기본값은 원본과 동일 → 제출 2·3 재현
#   4) 섹션 6·7은 main 에서만 실행 (experiments 에서 import 가능)
# 원본 파일은 읽기만 하고 수정하지 않습니다.
# ==========================================================================
import os, sys
import pandas as pd, numpy as np
import lightgbm as lgb
from sklearn.metrics import average_precision_score as ap
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# ---------- 0. 경로 · 설정 ----------
base = os.path.dirname(os.path.abspath(__file__))
DATA = f"{base}/epoch_data/epoch_data"
SNAP_GZ = f"{base}/video_snapshots_260901.csv.gz의 사본"
SNAP_PQ = f"{base}/video_snapshots.parquet"
OUT = f"{base}/submissions"; os.makedirs(OUT, exist_ok=True)

CFG = dict(
    USE_EARLY=True,        # 초반 조회수 피처(feats2)
    DECAY=0.7,             # 최근 기준일 가중 (1.0 = 가중 없음)
    PAST_OBS_ONLY=False,   # True면 log_past·n_long도 D 이전에 관측된 view_5d만 사용
    SELF_D=['2026-07-22', '2026-07-25', '2026-07-28', '2026-08-01', '2026-08-04', '2026-08-07'],
    OFF_D=['2026-08-11', '2026-08-25'],
    TRAIN_EXCLUDE=[],      # 라벨은 만들되 학습에서 뺄 기준일
    LD=18,                 # 자체 라벨 미래 구간(일). 연습 18 / 본선 26
)
USE_EARLY = CFG['USE_EARLY']

# ---------- 1. 기본 테이블 ----------
v  = pd.read_csv(f"{DATA}/video_5d_views.csv", parse_dates=['published_at'])
v['obs'] = v.published_at + pd.to_timedelta(v.h_after, unit='h')   # 5일차 조회수 실제 관측 시각
cs = pd.read_csv(f"{DATA}/channel_snapshots.csv", parse_dates=['collected_at'])
L  = v[v.is_shorts == 0]
START = pd.Timestamp('2026-07-09')

# ---------- 2. 라벨 (8/11 공식 라벨과 98.7% 일치) ----------
def label(D, Ld):
    D = pd.Timestamp(D)
    p = L[(L.published_at >= START) & (L.published_at < D)]
    f = L[(L.published_at >= D) & (L.published_at < D + pd.Timedelta(days=Ld))]
    ps = p.groupby('channel_id').view_5d.agg(['median', 'size'])
    ps = ps[(ps['size'] >= 3) & (ps['median'] >= 100)]
    fs = f.groupby('channel_id').view_5d.median().reindex(ps.index).fillna(0)
    g = np.log1p(fs) - np.log1p(ps['median'])
    return (g.rank(pct=True, ascending=False) <= 0.2).astype(int).rename('target'), g

# ---------- 3. 기본 피처 (기준일 D 이전 정보만) ----------
def feats(D, clean=True):
    D = pd.Timestamp(D)
    p = L[(L.published_at >= START) & (L.published_at < D)]
    known = p[p.obs < D] if clean else p            # 기준일 이후 관측된 view_5d 제외 (누수 방지)
    pp = known if CFG['PAST_OBS_ONLY'] else p       # 원본: log_past 는 관측시점 필터 없이 p 전체
    ps = pp.groupby('channel_id').view_5d.median()
    allv = v[v.published_at < D]
    c = cs[cs.collected_at < D]
    # --- 미래정보 assert ---
    assert known.obs.max() < D and allv.published_at.max() < D and c.collected_at.max() < D
    if CFG['PAST_OBS_ONLY']:
        assert pp.obs.max() < D
    F = pd.DataFrame(index=ps.index)
    F['log_past'] = np.log1p(ps)
    k = known.assign(lv=np.log1p(known.view_5d))
    for w in [7, 14]:
        r = k[k.published_at >= D - pd.Timedelta(days=w + 6)].groupby('channel_id').lv.median()
        F[f'rec{w}_minus_past'] = r.reindex(F.index) - F.log_past
    last3 = k.sort_values('published_at').groupby('channel_id').lv.apply(lambda s: s.tail(3).mean())
    F['last3_minus_past'] = last3.reindex(F.index) - F.log_past
    F['lv_std'] = k.groupby('channel_id').lv.std().reindex(F.index)
    F['lv_max_minus_past'] = k.groupby('channel_id').lv.max().reindex(F.index) - F.log_past
    F['n_long'] = pp.groupby('channel_id').size().reindex(F.index)
    for w in [7, 14, 28]:
        F[f'up_long_{w}'] = L[(L.published_at < D) & (L.published_at >= D - pd.Timedelta(days=w))].groupby('channel_id').size().reindex(F.index).fillna(0)
        F[f'up_all_{w}'] = allv[allv.published_at >= D - pd.Timedelta(days=w)].groupby('channel_id').size().reindex(F.index).fillna(0)
    F['days_since_long'] = (D - L[L.published_at < D].groupby('channel_id').published_at.max()).dt.total_seconds().reindex(F.index) / 86400
    F['shorts_ratio'] = allv.groupby('channel_id').is_shorts.mean().reindex(F.index)
    ks = k.assign(lr=np.log1p(k.like_5d) - k.lv, cr=np.log1p(k.comment_5d) - k.lv)
    F['like_ratio'] = ks.groupby('channel_id').lr.median().reindex(F.index)
    F['cmt_ratio'] = ks.groupby('channel_id').cr.median().reindex(F.index)
    last = c.sort_values('collected_at').groupby('channel_id').last()
    F['log_subs'] = np.log1p(last.subscriber_count.reindex(F.index))
    for w in [7, 14]:
        old = c[c.collected_at < D - pd.Timedelta(days=w)].sort_values('collected_at').groupby('channel_id').last()
        F[f'subs_g{w}'] = (np.log1p(last.subscriber_count) - np.log1p(old.subscriber_count)).reindex(F.index)
        F[f'tv_g{w}'] = (np.log1p(last.total_view_count) - np.log1p(old.total_view_count)).reindex(F.index)
    F['past_over_subs'] = F.log_past - F.log_subs
    return F

# ---------- 4. 초반 조회수 피처 (video_snapshots) ----------
early = None
def load_early():
    global early
    import duckdb
    con = duckdb.connect()
    if not os.path.exists(SNAP_PQ):   # 파일명이 .gz로 안 끝나서 compression 지정 필수
        con.sql(f"COPY (SELECT * FROM read_csv('{SNAP_GZ}', compression='gzip', header=true, delim=',')) "
                f"TO '{SNAP_PQ}' (FORMAT parquet, COMPRESSION zstd)")
    early = con.sql(f"""
      SELECT video_id, channel_id,
        MAX(view_count) FILTER (WHERE video_age_hours BETWEEN 20 AND 28) AS v24,
        MAX(view_count) FILTER (WHERE video_age_hours BETWEEN 44 AND 52) AS v48
      FROM read_parquet('{SNAP_PQ}') WHERE video_age_hours <= 52 GROUP BY 1, 2
    """).df()
    vids = pd.read_csv(f"{DATA}/videos.csv", usecols=['video_id', 'published_at', 'is_shorts'], parse_dates=['published_at'])
    early = early.merge(vids, on='video_id')
    early = early[early.is_shorts == 0]

def early_feats(D, F):
    D = pd.Timestamp(D)
    e = early[early.published_at + pd.Timedelta(hours=52) < D].copy()   # 48h 값이 D 전에 관측된 영상만
    assert (e.published_at + pd.Timedelta(hours=52)).max() < D          # 사용 스냅샷 시각 ≤ 게시+52h < D
    e['lv24'] = np.log1p(e.v24); e['lv48'] = np.log1p(e.v48); e['mom'] = e.lv48 - e.lv24
    rec = e[e.published_at >= D - pd.Timedelta(days=7)]
    ga, gr = e.groupby('channel_id'), rec.groupby('channel_id')
    out = pd.DataFrame(index=F.index)
    out['rec_lv24_vs_all'] = gr.lv24.median() - ga.lv24.median()
    out['rec_lv48_minus_past'] = gr.lv48.median() - F.log_past
    out['mom_rec'] = gr.mom.median(); out['mom_all'] = ga.mom.median()
    out['n_rec_early'] = gr.size()
    return out

def feats2(D):
    F = feats(D)
    return F.join(early_feats(D, F)) if CFG['USE_EARLY'] else F

# ---------- 5. 학습 데이터 ----------
tl = pd.read_csv(f"{DATA}/train_labels.csv"); tl['ch'] = tl.row_id.str[:-11]; tl['D'] = tl.row_id.str[-10:]
off = lambda D: tl[tl.D == D].set_index('ch').target
Y, Fr, Fk = {}, {}, {}
def build(dates):
    """기준일별 라벨·피처 구축 (D 이전 정보만)"""
    if CFG['USE_EARLY'] and early is None:
        load_early()
    for D in dates:
        Y[D] = off(D) if D in CFG['OFF_D'] else label(D, CFG['LD'])[0]
        Fr[D] = feats2(D).reindex(Y[D].index)
        Fk[D] = Fr[D].rank(pct=True)            # 기준일 안에서 순위 정규화

SMALL = ['log_past', 'last3_minus_past', 'lv_std', 'up_long_14', 'days_since_long', 'past_over_subs']
def lgbm(seed=0, mono=None):
    return lgb.LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=30,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=seed,
                              verbose=-1, monotone_constraints=mono)
def logit():
    return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(C=0.3, max_iter=2000))
def recency_w(dates, decay=None):   # 최근 기준일일수록 가중치 큼
    decay = CFG['DECAY'] if decay is None else decay
    n = len(dates)
    return np.concatenate([np.full(len(Y[d]), decay ** (n - 1 - i)) for i, d in enumerate(dates)])

def predict_all(train_dates, Xk_test, Xr_test, seeds=(0,)):
    """세 가지 방법의 예측을 반환 (index = 채널)"""
    Xk = pd.concat([Fk[d] for d in train_dates]); yy = pd.concat([Y[d] for d in train_dates])
    cols = list(Xk.columns); mono = [-1 if c == 'log_past' else 0 for c in cols]
    p1 = (-Xr_test.log_past).rank(pct=True)
    p2 = np.mean([lgbm(s).fit(Xk, yy, sample_weight=recency_w(train_dates)).predict_proba(Xk_test[cols])[:, 1] for s in seeds], axis=0)
    p2 = pd.Series(p2, Xk_test.index)
    pm = np.mean([lgbm(s, mono).fit(Xk, yy).predict_proba(Xk_test[cols])[:, 1] for s in seeds], axis=0)
    pl = logit().fit(Xk[SMALL], yy).predict_proba(Xk_test[SMALL])[:, 1]
    p3 = (pd.Series(pm, Xk_test.index).rank() + pd.Series(pl, Xk_test.index).rank() + p1.rank())
    p3 = p3.rank(pct=True)
    return {'01_logpast': p1, '02_lgbm_rank_f2': p2, '03_blend': p3}

def validate(val_dates=('2026-08-07', '2026-08-11', '2026-08-25'), seeds=(0,)):
    rows = []
    all_d = sorted(Y)
    for Dv in val_dates:
        tr = [d for d in all_d if d < Dv and d not in CFG['TRAIN_EXCLUDE']]
        preds = predict_all(tr, Fk[Dv], Fr[Dv], seeds=seeds)
        rows.append({'D': Dv, **{k: ap(Y[Dv], p.fillna(0.5)) for k, p in preds.items()}})   # 피처 없는 채널은 중간 순위
    val = pd.DataFrame(rows).set_index('D').T; val['mean'] = val.mean(axis=1)
    return val

def predict_test(D='2026-09-01', seeds=range(5)):
    sub = pd.read_csv(f"{DATA}/sample_submission.csv"); sub['ch'] = sub.row_id.str[:-11]
    Ft_raw = feats2(D).reindex(sub.ch.unique())
    n_missing = int(Ft_raw.log_past.isna().sum())
    final = predict_all([d for d in sorted(Y) if d not in CFG['TRAIN_EXCLUDE']], Ft_raw.rank(pct=True), Ft_raw, seeds=seeds)
    return sub, final, n_missing

def save_sub(sub, p, path):
    s = sub[['row_id']].copy()
    s['prediction'] = sub.ch.map(p).fillna(0.2).clip(0, 1).values
    ref = pd.read_csv(f"{DATA}/sample_submission.csv")
    assert len(s) == 694 == len(ref), len(s)
    assert s.prediction.notna().all() and s.prediction.between(0, 1).all()
    assert set(s.row_id) == set(ref.row_id) and s.row_id.is_unique
    assert not os.path.exists(path), f'이미 있음: {path}'
    s.to_csv(path, index=False)
    return s

if __name__ == '__main__':
    build(CFG['SELF_D'] + CFG['OFF_D'])
    print('검증 PR-AUC (무작위 ≈ 0.20)\n', validate().round(3))
    sub, final, nm = predict_test()
    print('제출 대상', len(sub), '| 피처 없는 채널', nm)
    for name, p in final.items():
        path = f"{os.environ.get('REPRO_DIR', OUT)}/repro_{name}.csv"
        if os.path.exists(path): os.remove(path)   # repro_ 는 재현 확인용 임시본
        save_sub(sub, p, path); print('저장:', path)

Writing epoch_pipeline_local.py


### 1-2. `structsim.py` — 구조 시뮬레이터

- `prep`: 채널별 평소 수준·영상별 변동 폭·업로드율·급등 크기·과거 중앙값 2개(9/1 기준, 8/25 기준)
- `posterior`: P(9/1 라벨 = 1 | 8/25 라벨 = 공식 값)

In [3]:
%%writefile structsim.py
# ==========================================================================
# 구조 시뮬레이터 (본선 B3~) — 8/26 이후 업로드 영상의 조회수 급등(전 채널 중앙값 약 +0.7 log)을 반영
#   1) 채널별 급등 크기 I_c 를 8/27~8/31 업로드 영상으로 추정 (경험적 베이즈 수축)
#   2) 9/1~9/26 업로드 영상을 채널별로 생성 (개수: 최근 28일 업로드율, 수준: 기준 + I_c + 잡음)
#   3) 같은 가상 세계에서 8/25 라벨(8/25~31 실제 영상 + 9/1~11 생성 영상)과 9/1 라벨을 함께 계산
#   4) P(9/1 라벨=1 | 8/25 공식 라벨=관측값) 을 채널별로 집계
# 9/1 이후 관측된 view_5d(8/27~31 업로드분)를 사용한다 — 운영진 답변(10/2)상 미금지, 발표자료에 명시할 것
# ==========================================================================
import pandas as pd, numpy as np
from sklearn.metrics import average_precision_score as ap
import os
BASE=os.path.dirname(os.path.abspath(__file__))
DATA=BASE+'/epoch_data/epoch_data/'
v=pd.read_csv(DATA+'video_5d_views.csv',parse_dates=['published_at'])
v['obs']=v.published_at+pd.to_timedelta(v.h_after,unit='h')
L=v[v.is_shorts==0].copy(); L['lv']=np.log1p(L.view_5d)
START=pd.Timestamp('2026-07-09')
tl=pd.read_csv(DATA+'train_labels.csv'); tl['ch']=tl.row_id.str[:-11]; tl['D']=tl.row_id.str[-10:]
off=lambda D: tl[tl.D==D].set_index('ch').target
T=pd.Timestamp; dd=lambda n: pd.Timedelta(days=n)
def past(D):
    D=T(D); p=L[(L.published_at>=START)&(L.published_at<D)]
    ps=p.groupby('channel_id').view_5d.agg(['median','size'])
    return ps[(ps['size']>=3)&(ps['median']>=100)]
def growth(D,h):
    D=T(D); ps=past(D)
    f=L[(L.published_at>=D)&(L.published_at<D+dd(h))]
    fs=f.groupby('channel_id').view_5d.median().reindex(ps.index).fillna(0)
    return np.log1p(fs)-np.log1p(ps['median'])
def label(D,h):
    g=growth(D,h); return (g.rank(pct=True,ascending=False)<=0.2).astype(int), g

# 구조 시뮬레이터: 채널별 '미래 영상'을 생성해 (직전 라벨, 테스트 라벨)을 함께 모사하고
# P(테스트 라벨=1 | 직전 라벨=관측값) 을 계산한다.
def med_rows(X):
    """행별 중앙값(NaN 무시, raw 조회수 기준 중앙값의 log1p). 전부 NaN이면 0(=log1p(0))"""
    Xs=np.sort(X,axis=1); cnt=(~np.isnan(X)).sum(1); r=np.arange(len(X))
    lo=Xs[r,np.maximum((cnt-1)//2,0)]; hi=Xs[r,np.maximum(cnt//2,0)]
    m=np.log1p((np.expm1(lo)+np.expm1(hi))/2)
    return np.where(cnt==0,0.0,m)

def prep(D,lag,sample_D,sample_lag,shift_from=None,rate_win=28,k_s=5):
    D,lag=T(D),T(lag); sf=T(shift_from) if shift_from else lag
    P=L[(L.published_at>=START)&(L.published_at<D)]
    chs=sorted(set(sample_D)|set(sample_lag))
    lpD=np.log1p(P.groupby('channel_id').view_5d.median())
    lpL=np.log1p(P[P.published_at<lag].groupby('channel_id').view_5d.median())
    B=P[P.published_at<sf]; base=B.groupby('channel_id').lv.median()
    res=(B.lv-B.channel_id.map(base))
    mad=res.abs().groupby(B.channel_id).median()*1.4826; nb=B.groupby('channel_id').size()
    s0=float(mad[nb>=5].median())
    s=np.sqrt((nb*mad**2+k_s*s0**2)/(nb+k_s))
    Vs=P[P.published_at>=sf]; xbar=(Vs.lv-Vs.channel_id.map(base)).groupby(Vs.channel_id).mean(); nv=Vs.groupby('channel_id').size()
    # 경험적 베이즈 사전분포
    ok=xbar.dropna().index; se2=(s.reindex(ok)**2/nv.reindex(ok))
    mu=float(np.median(xbar[ok])); sigI2=max(float(np.var(xbar[ok].clip(mu-3,mu+3))-se2.median()),0.02)
    n28=P[P.published_at>=D-dd(rate_win)].groupby('channel_id').size()
    Vlag={c:g.lv.values for c,g in P[P.published_at>=lag].groupby('channel_id')}
    out=[]
    for c in chs:
        b=base.get(c,np.nan)
        if np.isnan(b): b=lpD.get(c,np.nan)   # 기준 구간 영상이 없으면 전체 중앙값
        sc=s.get(c,s0); 
        if not np.isfinite(sc): sc=s0
        n=nv.get(c,0); xb=xbar.get(c,mu) if n>0 else mu
        pv=1/(n/sc**2+1/sigI2); pm=pv*(xb*n/sc**2+mu/sigI2)
        out.append(dict(ch=c,base=b,s=sc,pm=pm,pv=pv,n28=n28.get(c,0),lpD=lpD.get(c,np.nan),lpL=lpL.get(c,np.nan),V=Vlag.get(c,np.array([])),nv=n))
    return pd.DataFrame(out).set_index('ch'),dict(mu=mu,sigI=np.sqrt(sigI2),s0=s0)

def simulate(prep_df,hp,sample_D,sample_lag,h,ov=11,S=2000,rho=1.0,sig_eta=0.3,rate_win=28,tdf=None,seed=0,s_mult=1.0,drift=0.0):
    rng=np.random.default_rng(seed); chs=list(prep_df.index); n=len(chs)
    GD=np.empty((n,S)); GL=np.empty((n,S))
    for i,(c,r) in enumerate(prep_df.iterrows()):
        I=rng.normal(r.pm,np.sqrt(r.pv),S)
        delta=hp['mu']+rho*(I-hp['mu'])+rng.normal(0,sig_eta,S)+drift
        lam=rng.gamma(r.n28+0.5,1/rate_win,S); N=rng.poisson(lam*h); Nmax=max(int(N.max()),1)
        eps=rng.standard_normal((S,Nmax)) if tdf is None else rng.standard_t(tdf,(S,Nmax))*np.sqrt((tdf-2)/tdf)
        X=r.base+delta[:,None]+r.s*s_mult*eps
        mask=np.arange(Nmax)[None,:]<N[:,None]
        XF=np.where(mask,X,np.nan)
        first=rng.random((S,Nmax))<(ov/h)
        XH=np.where(mask&first,X,np.nan)
        if len(r.V): XH=np.concatenate([XH,np.broadcast_to(r.V,(S,len(r.V)))],axis=1)
        GD[i]=med_rows(XF)-r.lpD; GL[i]=med_rows(XH)-r.lpL
    idx=pd.Index(chs); iD=idx.isin(sample_D); iL=idx.isin(sample_lag)
    def top(G,m):
        out=np.zeros_like(G,dtype=bool); sub=G[m]; k=int(np.floor(0.2*sub.shape[0]))
        thr=-np.sort(-sub,axis=0)[k-1]; out[m]=sub>=thr; return out
    YD=top(GD,iD); YL=top(GL,iL)
    return idx,YD,YL,iD,iL

def posterior(idx,YD,YL,iL,lab,a=5):
    """P(YD=1 | YL=관측 라벨). lab: Series(ch→0/1), 없는 채널은 무조건부"""
    pu=YD.mean(1); pl=YL.mean(1); out=pu.copy(); ll=0.0
    for i,c in enumerate(idx):
        if iL[i] and c in lab.index:
            m=YL[i]==bool(lab[c]); out[i]=(YD[i][m].sum()+a*pu[i])/(m.sum()+a)
            ll+=np.log(np.clip(pl[i] if lab[c] else 1-pl[i],1e-3,1))
    return pd.Series(out,idx),pd.Series(pu,idx),pd.Series(pl,idx),ll

Writing structsim.py


### 1-3. `simp.py` — 급등 사전분포를 쓰는 시뮬레이터

- `prep_prior`: 급등 크기 = 가중평균(실제로 잰 값, 채널별 예상값). `r2`(사전분포 신뢰도)가 클수록 예상값 쪽으로 당김
- `simulate_p`: 가상의 9/1~9/26 영상을 생성해 같은 세계에서 9/1 라벨과 8/25 라벨을 함께 계산

In [4]:
%%writefile simp.py
from structsim import *
def prep_prior(pdf,hp,prior,r2):
    """사전평균을 채널별 회귀 예측으로 교체하고 사후분포 재계산"""
    d=pdf.copy(); mu_c=prior.reindex(d.index).fillna(hp['mu']); sig2=hp['sigI']**2*(1-r2)
    Vs=L[L.published_at>='2026-08-27']; base=d.base
    xb=(Vs.lv-Vs.channel_id.map(base)).groupby(Vs.channel_id).mean().reindex(d.index)
    n=d.nv; xb=xb.where(n>0,mu_c)
    d['pv']=1/(n/d.s**2+1/sig2); d['pm']=d.pv*(xb*n/d.s**2+mu_c/sig2); d['mu_c']=mu_c
    return d
def simulate_p(prep_df,sample_D,sample_lag,h,ov=11,S=2000,rho=1.0,sig_eta=0.3,rate_win=28,seed=0,s_mult=1.0):
    rng=np.random.default_rng(seed); chs=list(prep_df.index); n=len(chs)
    GD=np.empty((n,S)); GL=np.empty((n,S))
    for i,(c,r) in enumerate(prep_df.iterrows()):
        I=rng.normal(r.pm,np.sqrt(r.pv),S)
        delta=r.mu_c+rho*(I-r.mu_c)+rng.normal(0,sig_eta,S)
        lam=rng.gamma(r.n28+0.5,1/rate_win,S); N=rng.poisson(lam*h); Nmax=max(int(N.max()),1)
        X=r.base+delta[:,None]+r.s*s_mult*rng.standard_normal((S,Nmax))
        mask=np.arange(Nmax)[None,:]<N[:,None]
        XF=np.where(mask,X,np.nan); first=rng.random((S,Nmax))<(ov/h); XH=np.where(mask&first,X,np.nan)
        if len(r.V): XH=np.concatenate([XH,np.broadcast_to(r.V,(S,len(r.V)))],axis=1)
        GD[i]=med_rows(XF)-r.lpD; GL[i]=med_rows(XH)-r.lpL
    idx=pd.Index(chs); iD=idx.isin(sample_D); iL=idx.isin(sample_lag)
    def top(G,m):
        out=np.zeros_like(G,dtype=bool); sub=G[m]; k=int(np.floor(0.2*sub.shape[0]))
        thr=-np.sort(-sub,axis=0)[k-1]; out[m]=sub>=thr; return out
    return idx,top(GD,iD),top(GL,iL),iD,iL

Writing simp.py


### 1-4. `make_b26_funcs.py` — 기존 영상 배율과 급등 사전분포

- `old_video_multiplier`: 8/22 이전 업로드 롱폼의 일 조회 증가율을 급등 전후로 비교(같은 영상의 이중차분, 나이 효과 제거)
- `surge_prior_plus`: 채널 특성 + 기존 영상 배율 → 급등 크기 회귀(LightGBM + 릿지 평균, OOF 예측)

In [5]:
%%writefile make_b26_funcs.py
# make_b26.py 의 함수 정의부 사본(실행·저장 없음). make_b27_b29.py 가 불러 쓴다
# 본선 후보 B26 (python make_b26.py, 약 2분) — B8 의 급등 사전분포에 '기존 영상 배율' 피처를 추가한 것. 결합 방식은 B8 과 같다.
#   기존 영상 배율: 8/22 이전 업로드 롱폼의 일 조회 증가율을 [8/22 12시~8/26 23시] 와 [8/27 12시~8/31 23시] 로 비교(같은 영상의 이중차분).
#     나이에 따른 자연 감소는 열흘 앞 같은 구조의 창([8/12~8/16] → [8/17~8/21])의 나이대별 중앙값으로 뺀다(쇼츠 포함 전체 영상 기준).
#     급등(8/27 낮 시작)은 달력 시간 기준이라 기존 롱폼의 조회 증가도 +0.4~0.8 log 올랐다(좋아요·쇼츠는 변화 없음).
#     8/27 이후 업로드 영상과 겹치지 않고, 8/31 23:45 까지의 스냅샷만 쓴다. 694채널 중 676개, 급등 영상 0~1개 302채널 중 288개에서 계산된다.
#   검증: 사전분포 OOF R2 0.211 → 0.285 (fold 시드 5개 평균 0.216 → 0.279, 사전에 정한 기준 0.30 에는 미달).
#         8/25 공식 라벨(미사용) 적중 AP 0.808 → 0.814, 로그우도 −189.1 → −187.8.
import numpy as np, pandas as pd, lightgbm as lgb, warnings; warnings.filterwarnings('ignore')
import pyarrow.parquet as pq, pyarrow.compute as pc, pyarrow as pa
import epoch_pipeline_local as P
from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
SF='2026-08-27'; TS=pd.Timestamp
ch_=pd.read_csv(P.DATA+'/sample_submission.csv').row_id.str[:-11].values

def old_video_multiplier():
    vid=pd.read_csv(P.DATA+'/videos.csv',usecols=['video_id','channel_id','published_at','is_shorts'],parse_dates=['published_at']).set_index('video_id')
    lf=vid[vid.channel_id.isin(ch_)&(vid.published_at>='2026-07-12')&(vid.published_at<SF)]
    tb=pq.read_table(P.base+'/video_snapshots.parquet',columns=['video_id','collected_at','view_count'])
    tb=tb.filter(pc.is_in(tb['video_id'],value_set=pa.array(lf.index.values))); d=tb.to_pandas().sort_values(['video_id','collected_at'])
    assert d.collected_at.max()<TS('2026-09-01')
    at=lambda t: d[(d.collected_at<=TS(t))&(d.collected_at>TS(t)-pd.Timedelta(hours=30))].groupby('video_id').tail(1).set_index('video_id')
    def rate(t0,t1):   # 일 조회 증가율(log), 창 시작 시점 나이(일)
        a=at(t0); b=at(t1); k=a.index.intersection(b.index); hrs=(b.collected_at[k]-a.collected_at[k]).dt.total_seconds()/3600
        o=pd.DataFrame({'rv':np.log((b.view_count[k]-a.view_count[k]).clip(lower=0)/hrs*24+1),'age':(TS(t0)-lf.published_at.reindex(k)).dt.total_seconds()/86400})
        return o[(hrs>48)&(o.age>0.5)]
    def did(t0,t1,t2,t3):
        a=rate(t0,t1); b=rate(t2,t3); k=a.index.intersection(b.index)
        o=pd.DataFrame({'dv':b.rv[k]-a.rv[k],'age':b.age[k]}); o['ab']=pd.cut(o.age,[0.5,1,2,3,5,7,10,14,21,30,60]); return o
    post=did('2026-08-22 12:00','2026-08-26 23:00','2026-08-27 12:00','2026-08-31 23:00')
    ref=did('2026-08-12 12:00','2026-08-16 23:00','2026-08-17 12:00','2026-08-21 23:00').groupby('ab').dv.median()
    post['ev']=post.dv-post.ab.map(ref).astype(float); post['ch']=lf.channel_id.reindex(post.index)
    g=post[lf.is_shorts.reindex(post.index)==0].groupby('ch')
    return pd.DataFrame({'did_ev':g.ev.mean().clip(-3,4),'did_med':g.ev.median().clip(-3,4),'did_n':g.size()})

def surge_prior_plus(extra):
    """surge_prior.py 와 같은 절차 + 추가 피처"""
    P.CFG['USE_EARLY']=False; P.CFG['PAST_OBS_ONLY']=True
    L=P.L.assign(lv=np.log1p(P.L.view_5d)); F=P.feats(SF)
    chn=pd.read_csv(P.DATA+'/channels.csv',usecols=['channel_id','country','created_at']).set_index('channel_id')
    F['kr']=(chn.country.reindex(F.index)=='KR').astype(float); F['age']=(pd.Timestamp('2026-09-01')-pd.to_datetime(chn.created_at.reindex(F.index)).dt.tz_localize(None)).dt.days
    base=L[L.published_at<SF].groupby('channel_id').lv.median()
    V=L[L.published_at>=SF]; xbar=(V.lv-V.channel_id.map(base)).groupby(V.channel_id).mean(); nv=V.groupby('channel_id').size()
    F=F.reindex(ch_).join(extra.reindex(ch_)); y=xbar.reindex(ch_); n=nv.reindex(ch_).fillna(0)
    m=(n>=2).values&F.log_past.notna().values
    X=F[m]; yy=y[m].clip(-1.5,3); w=np.minimum(n[m],6)
    mk_l=lambda: lgb.LGBMRegressor(n_estimators=200,learning_rate=0.03,num_leaves=7,min_child_samples=20,subsample=0.8,subsample_freq=1,colsample_bytree=0.7,verbose=-1,random_state=0)
    mk_r=lambda: make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),RidgeCV(alphas=[1,10,100]))
    oof=np.zeros(m.sum())
    for tr,te in KFold(5,shuffle=True,random_state=0).split(X):
        oof[te]=(mk_l().fit(X.iloc[tr],yy.iloc[tr],sample_weight=w.iloc[tr]).predict(X.iloc[te])+mk_r().fit(X.iloc[tr],yy.iloc[tr],ridgecv__sample_weight=w.iloc[tr]).predict(X.iloc[te]))/2
    r2=1-np.sum(w*(yy-oof)**2)/np.sum(w*(yy-np.average(yy,weights=w))**2)
    ok=F.log_past.notna().values; full=pd.Series(np.nan,index=ch_)
    full[ok]=(mk_l().fit(X,yy,sample_weight=w).predict(F[ok])+mk_r().fit(X,yy,ridgecv__sample_weight=w).predict(F[ok]))/2
    full[ch_[m]]=oof; return full,float(r2)

Writing make_b26_funcs.py


## 2. E — LightGBM 분류

- 학습 기준일 8개: 자체 계산 라벨 6개(7/22, 7/25, 7/28, 8/1, 8/4, 8/7) + 공식 라벨 2개(8/11, 8/25)
- 초반 조회수 입력(업로드 후 24·48시간)은 스냅샷에서 pyarrow 로 만듭니다(원본 `load_early` 는 duckdb 사용, 같은 값).
- `PAST_OBS_ONLY=True`: 기준일 이전에 관측된 view_5d 만 씁니다.

In [6]:
import epoch_pipeline_local as P
import pyarrow.dataset as ds

def load_early_pa():
    """(video_id, channel_id)별 v24 = 20~28시간 최대 조회수, v48 = 44~52시간 최대 조회수 (롱폼만)"""
    t = ds.dataset(P.SNAP_PQ).to_table(columns=['video_id', 'channel_id', 'video_age_hours', 'view_count'],
                                       filter=ds.field('video_age_hours') <= 52).to_pandas()
    a = t.video_age_hours
    t['v24'] = t.view_count.where((a >= 20) & (a <= 28))
    t['v48'] = t.view_count.where((a >= 44) & (a <= 52))
    e = t.groupby(['video_id', 'channel_id'], sort=False)[['v24', 'v48']].max().reset_index()
    vids = pd.read_csv(f"{P.DATA}/videos.csv", usecols=['video_id', 'published_at', 'is_shorts'], parse_dates=['published_at'])
    e = e.merge(vids, on='video_id')
    P.early = e[e.is_shorts == 0]
P.load_early = load_early_pa

P.CFG['PAST_OBS_ONLY'] = True; P.CFG['USE_EARLY'] = True; P.CFG['LD'] = 18
P.Y.clear(); P.Fr.clear(); P.Fk.clear()
P.build(P.CFG['SELF_D'] + P.CFG['OFF_D'])
print('학습 기준일', sorted(P.Y)); print('학습 행 수', sum(len(y) for y in P.Y.values()), '| 입력 수', P.Fk['2026-08-25'].shape[1])

학습 기준일 ['2026-07-22', '2026-07-25', '2026-07-28', '2026-08-01', '2026-08-04', '2026-08-07', '2026-08-11', '2026-08-25']
학습 행 수 3730 | 입력 수 28


### 2-1. hold-out 검증 (PR-AUC, 무작위 기준 약 0.20)

검증 기준일보다 앞선 기준일만으로 학습해 그 기준일을 맞힙니다. 8/25는 급등 구간이라 E 단독으로는 낮습니다. 이것이 시뮬레이터를 만든 이유입니다.

In [7]:
val = P.validate(seeds=tuple(range(5)))
print(val.round(4).to_string())

D                2026-08-07  2026-08-11  2026-08-25    mean
01_logpast           0.2806      0.3071      0.3982  0.3287
02_lgbm_rank_f2      0.5373      0.4663      0.2932  0.4323
03_blend             0.3972      0.4290      0.3691  0.3984


In [8]:
sub_e, final, n_missing = P.predict_test()            # 9/1 예측, 5시드 평균
e_path = f'{P.OUT}/nb_E_lgbm_pastobsonly.csv'
if os.path.exists(e_path): os.remove(e_path)
P.save_sub(sub_e, final['02_lgbm_rank_f2'], e_path)
print('저장', os.path.relpath(e_path), '| 제출 대상', len(sub_e), '| 피처 없는 채널', n_missing)

저장 submissions/nb_E_lgbm_pastobsonly.csv | 제출 대상 694 | 피처 없는 채널 1


## 3. 방법 ① B1 — 8/25 라벨 + 숨은 구간 역산

8/25 공식 라벨은 8/25~9/11 영상으로 정해졌습니다. 그중 8/25~8/31은 데이터에 보이고, 9/1~9/11은 테스트 구간과 겹치는 숨은 부분입니다.

- 라벨이 Rising인데 보이는 부분이 약했다 → 숨은 부분이 강했다 → 점수 올림
- 라벨이 Rising이 아닌데 보이는 부분이 강했다 → 숨은 부분이 약했다 → 점수 내림

`B1 = rank(E) + 0.5 × (8/25 라벨) − 0.25 × rank(8/25~31 영상의 성장)`

In [9]:
D_TEST, LAG_D, W_LAB, W_HID = '2026-09-01', '2026-08-25', 0.5, 0.25

def hidden_rank(D, lag_d, index):
    """직전 라벨일~D 업로드 영상의 view_5d 중앙값 성장(라벨일 과거 중앙값 대비)의 날짜 내 순위"""
    D, lag_d = pd.Timestamp(D), pd.Timestamp(lag_d); L = P.L
    past = L[(L.published_at >= P.START) & (L.published_at < lag_d)].groupby('channel_id').view_5d.median()
    k = L[(L.published_at >= lag_d) & (L.published_at < D)]
    assert k.published_at.max() < D                # 업로드는 모두 D 이전 (view_5d 관측은 D 이후 포함)
    g = (np.log1p(k.groupby('channel_id').view_5d.median()) - np.log1p(past)).reindex(index)
    return g.rank(pct=True).fillna(0.5), int(g.notna().sum())

pe = final['02_lgbm_rank_f2'].rank(pct=True)
lab = P.off(LAG_D).reindex(pe.index); assert lab.dropna().isin([0, 1]).all()
rg, n_hid = hidden_rank(D_TEST, LAG_D, pe.index)
score = pe + W_LAB * lab.fillna(0.2) - W_HID * rg          # 라벨 없는 채널은 0.2, 보이는 영상이 없으면 순위 0.5
score = (score - score.min()) / (score.max() - score.min())
b1_path = f'{P.OUT}/nb_B1_lgbm_lag0825_hidden.csv'
if os.path.exists(b1_path): os.remove(b1_path)
P.save_sub(sub_e, score, b1_path)
print('저장', os.path.relpath(b1_path), '| 라벨 커버', int(lab.notna().sum()), '| 보정 커버', n_hid)

저장 submissions/nb_B1_lgbm_lag0825_hidden.csv | 라벨 커버 666 | 보정 커버 597


## 4. view_5d 재계산 — 코드북 정의(108~132시간 구간 최댓값)

배포된 `video_5d_views.csv` 는 120시간 시점의 보간값(31,524행)입니다. 코드북의 정의는 "업로드 후 108~132시간 구간 스냅샷 조회수의 최댓값"(31,796행)입니다. 스냅샷에서 정의대로 다시 계산하면 공식 표본과 라벨이 재현됩니다.

- 8/26 12시 이후 업로드 영상은 구간이 9/1을 넘어 스냅샷이 없으므로 배포 표의 값을 씁니다.
- 스냅샷은 8/31 23:59까지만 씁니다(assert).

In [10]:
import structsim, simp
from structsim import *      # START, tl, off, T, dd, posterior, DATA, BASE ...

def official_like_L():
    """롱폼 view_5d 표를 코드북 정의(108~132시간 구간 최댓값)로 다시 만든다"""
    vid = pd.read_csv(DATA + 'videos.csv', usecols=['video_id', 'channel_id', 'published_at', 'is_shorts'], parse_dates=['published_at'])
    v5 = pd.read_csv(DATA + 'video_5d_views.csv', usecols=['video_id', 'view_5d']).rename(columns={'view_5d': 'v_dist'})
    f = (ds.field('video_age_hours') >= 108) & (ds.field('video_age_hours') <= 132)
    w = ds.dataset(BASE + '/video_snapshots.parquet').to_table(columns=['video_id', 'collected_at', 'view_count'], filter=f).to_pandas()
    assert w.collected_at.max() < pd.Timestamp('2026-09-01')
    vmax = w.dropna(subset=['view_count']).groupby('video_id').view_count.max().rename('v_max')   # 조회수 null 스냅샷 제외
    a = vid.merge(v5, on='video_id', how='left').merge(vmax, left_on='video_id', right_index=True, how='left')
    late = a.published_at >= pd.Timestamp('2026-08-26 12:00')        # 구간 끝(132h)이 9/1 0시를 넘는 영상: 배포 값도 사용
    a['view_5d'] = np.where(a.v_max.notna(), np.fmax(a.v_max, np.where(late, a.v_dist, np.nan)), np.where(late, a.v_dist, np.nan))
    a = a.dropna(subset=['view_5d'])
    print('다시 만든 표 행 수', len(a), '(코드북 31,796 / 배포 31,524) | 배포 표에 없던 영상', int(a.v_dist.isna().sum()))
    lf = a[a.is_shorts == 0].copy(); lf['lv'] = np.log1p(lf.view_5d)
    return lf[['video_id', 'channel_id', 'published_at', 'view_5d', 'is_shorts', 'lv']]

sub = pd.read_csv(DATA + 'sample_submission.csv'); sub['ch'] = sub.row_id.str[:-11]
L_old = structsim.L; L_new = official_like_L()
for name, LL in [('배포 표', L_old), ('다시 만든 표', L_new)]:          # 공식 표본·라벨 재현 비교
    structsim.L = LL; out = []
    for D in ['2026-08-11', '2026-08-25', '2026-09-01']:
        offc = set(tl[tl.D == D].ch) if D != '2026-09-01' else set(sub.ch); mine = set(structsim.past(D).index)
        out.append(f'{D[5:]} {len(offc & mine)}/{len(offc)}')
    y11, _ = structsim.label('2026-08-11', 18); o11 = off('2026-08-11'); kk = o11.index.intersection(y11.index)
    print(f'{name}: 공식 표본 재현', ' · '.join(out), '| 8/11 라벨 불일치', int((y11[kk] != o11[kk]).sum()), '/', len(kk))
structsim.L = L_new; simp.L = L_new          # 시뮬레이터는 다시 만든 표를 읽는다

다시 만든 표 행 수 31794 (코드북 31,796 / 배포 31,524) | 배포 표에 없던 영상 282
배포 표: 공식 표본 재현 08-11 596/601 · 08-25 662/666 · 09-01 688/694 | 8/11 라벨 불일치 8 / 596
다시 만든 표: 공식 표본 재현 08-11 601/601 · 08-25 666/666 · 09-01 694/694 | 8/11 라벨 불일치 3 / 601


## 5. 급등 크기 사전분포

8/27부터 롱폼 조회수가 전 채널에서 약 2배가 됐습니다(집계 기준 변경). 채널마다 오른 정도가 다르고, 8/27~31에 영상이 0~1편뿐인 채널(302개)은 직접 잴 수 없어 예상값을 만듭니다.

- 기존 영상 배율: 예전에 올린 영상도 8/27부터 조회수가 더 빨리 늘었으므로, 같은 영상의 일 조회 증가율을 급등 전후로 비교합니다.
- 회귀: 급등 영상 2편 이상인 채널로 학습, 학습에 쓴 채널에는 OOF 예측을 넣습니다.

In [11]:
import make_b26_funcs as M
feat = M.old_video_multiplier()
prior, r2 = M.surge_prior_plus(feat)
print('기존 영상 배율이 계산된 채널', int(feat.reindex(sub.ch.values).did_ev.notna().sum()), '/ 694')
print('급등 사전분포 OOF R2', round(r2, 3))

기존 영상 배율이 계산된 채널 676 / 694
급등 사전분포 OOF R2 0.285


## 6. 방법 ② 구조 시뮬레이터

1. 채널의 급등 크기를 사후분포(실제로 잰 값과 예상값의 가중평균, 사전분포 신뢰도 0.6)에서 뽑습니다.
2. 9월 수준 = 예상값 + 지속성 × (뽑은 값 − 예상값) + 채널 잡음(0.3). 지속성은 0.5와 0.75를 2회씩.
3. 9/1~9/26 영상 개수를 최근 28일 업로드율로 뽑고, 영상마다 조회수를 생성합니다(영상 잡음 = 채널 고유 폭 × 1.3).
4. 같은 가상 세계에서 9/1 라벨(과거 중앙값은 정확히 계산)과 8/25 라벨(실제 8/25~31 영상 + 생성 영상 중 9/1~9/11분)을 계산합니다.
5. 8/25 라벨이 공식 정답과 같게 나온 세계만 남겨 P(9/1 Rising)을 집계합니다.

가상 세계 16,000개 × 4회(시드 500~503)입니다. 8/25 공식 라벨을 보지 않고 맞힌 점수(blind)를 함께 출력합니다.

In [12]:
from sklearn.metrics import average_precision_score as AP, log_loss
o = off('2026-08-25')                                     # 8/25 공식 라벨
pdf, hp = structsim.prep('2026-09-01', '2026-08-25', sub.ch.values, o.index, shift_from='2026-08-27'); pdf['lpL'] = pdf.lpL.fillna(pdf.lpD)
assert pdf.lpD.reindex(sub.ch.values).notna().all()       # 9/1 과거 중앙값이 694채널 모두 계산됨
d = simp.prep_prior(pdf, hp, prior, 0.6); posts = []; pls = []
for j, rho in enumerate([0.5, 0.75, 0.5, 0.75]):
    idx, YD, YL, iD, iL = simp.simulate_p(d, sub.ch.values, o.index, 26, S=16000, rho=rho, sig_eta=0.3, s_mult=1.3, seed=500 + j)
    po, pu, pl, ll = posterior(idx, YD, YL, iL, o); posts.append(po.reindex(sub.ch)); pls.append(pl)
    print(f'시뮬레이션 {j + 1}/4 (지속성 {rho}) 완료', flush=True)
k = o.index.intersection(pd.Index(sub.ch.values))
pl = pd.concat(pls, axis=1).mean(axis=1)[k].clip(1e-3, 1 - 1e-3)
print(f'8/25 라벨 blind: AP {AP(o[k].values, pl):.3f}, 로그우도 {-log_loss(o[k].values, pl, normalize=False):.1f}')
sim = pd.concat(posts, axis=1).mean(axis=1)               # 채널별 P(9/1 Rising)
print('시뮬레이터 평균 확률: 8/25 Rising 채널', round(sim[o.reindex(sim.index) == 1].mean(), 3), '| 아닌 채널', round(sim[o.reindex(sim.index) == 0].mean(), 3))

시뮬레이션 1/4 (지속성 0.5) 완료


시뮬레이션 2/4 (지속성 0.75) 완료


시뮬레이션 3/4 (지속성 0.5) 완료


시뮬레이션 4/4 (지속성 0.75) 완료


8/25 라벨 blind: AP 0.820, 로그우도 -185.8
시뮬레이터 평균 확률: 8/25 Rising 채널 0.603 | 아닌 채널 0.098


## 7. 결합과 저장

`B31 = 0.5 × rank(시뮬레이터 확률) + 0.5 × rank(B1)` 을 0~1로 맞춥니다. 평가지표(AP)는 순서만 보므로 순위로 섞습니다.

In [13]:
rd = lambda path: pd.Series(pd.read_csv(path).prediction.values, index=sub.ch)
mm = lambda x: (x - x.min()) / (x.max() - x.min()); rk = lambda x: x.rank(pct=True)
b1 = rd(b1_path)
p = mm(0.5 * rk(sim) + 0.5 * rk(b1))
res = sub[['row_id']].copy(); res['prediction'] = p.reindex(sub.ch).values
assert len(res) == 694 and res.prediction.notna().all() and res.prediction.between(0, 1).all() and res.row_id.is_unique
out_path = 'submissions/nb_submit_b31.csv'; res.to_csv(out_path, index=False)
top = p.nlargest(139).index
print('저장', out_path, '| 행', len(res))
print('시뮬레이터와 B1의 순위상관', round(sim.corr(b1, method='spearman'), 3))
print('상위 139 중 8/25 Rising', int((o.reindex(top) == 1).sum()), '| 아님', int((o.reindex(top) == 0).sum()), '| 라벨 없음', int(o.reindex(top).isna().sum()))

저장 submissions/nb_submit_b31.csv | 행 694
시뮬레이터와 B1의 순위상관 0.646
상위 139 중 8/25 Rising 121 | 아님 11 | 라벨 없음 7


## 8. 제출 파일과의 대조

`ref/` 폴더에 리더보드에 낸 파일이 있으면 이 노트북의 결과와 비교합니다(없으면 건너뜀).

In [14]:
def compare(a, path, name):
    if not os.path.exists(path): print(name, ': 기준 파일 없음 →', path); return
    b = rd(path); T_ = lambda z, n: set(z.nlargest(n).index)
    print(f'{name}: 순위상관 {a.corr(b, method="spearman"):.5f} | 최대 절대차 {np.abs(a - b).max():.2e} | 상위 139 겹침 {len(T_(a, 139) & T_(b, 139))} | 상위 30 겹침 {len(T_(a, 30) & T_(b, 30))}')
compare(rd(e_path), 'ref/submit_04_lgbm_pastobsonly.csv', 'E')
compare(b1, 'ref/submit_07_lgbm_lag0825_hidden_full.csv', 'B1')
compare(p, 'ref/submit_b31_v5fix_S16k_r06_x_b1.csv', 'B31(최종 제출)')
print(f'전체 실행 시간 {time.time() - T0:.0f}초')

E: 순위상관 1.00000 | 최대 절대차 1.46e-03 | 상위 139 겹침 139 | 상위 30 겹침 30
B1: 순위상관 1.00000 | 최대 절대차 0.00e+00 | 상위 139 겹침 139 | 상위 30 겹침 30
B31(최종 제출): 순위상관 1.00000 | 최대 절대차 1.11e-16 | 상위 139 겹침 139 | 상위 30 겹침 30
전체 실행 시간 112초


## 참고

- 리더보드 점수(본선, PR-AUC): B1 0.6913 → 시뮬레이터 단독 0.6755 → 순위 평균 0.7369 → 급등 사전분포 0.7381 → 기존 영상 배율 0.7437 → 사전분포 신뢰도 0.6 0.7449 → view_5d 재계산 **0.7466**.
- 사전분포 신뢰도 0.6과 결합 비중 0.5는 리더보드 점수를 보고 고른 값입니다. 채널별 정답을 리더보드로 역산하지 않았습니다.
- 694행이라 ±0.03은 잡음입니다. 순위 평균 이후의 개선(+0.0097)은 통계적으로 확인된 것이 아닙니다.
- E의 초반 조회수 입력은 원본에서는 duckdb 로 만들었고 이 노트북은 pyarrow 로 만듭니다. 같은 정의이며 E는 1개 채널에서만 예측값이 1.5e-3 다릅니다. B1과 최종 제출 파일은 리더보드에 낸 파일과 같습니다(8절의 대조 결과).